# Phase 4: Portfolio Construction & Risk Management  
## Notebook 04_10 — Hierarchical Risk Parity

### Research question

How can hierarchical clustering and recursive risk allocation produce diversified portfolios without directly inverting a noisy covariance matrix?

This notebook follows:

```text
04_04_mean_variance_optimization_ledoit_wolf
04_05_pca_spectral_risk_analysis
04_09_risk_parity_and_equal_risk_contribution
```

The previous notebook built Equal Risk Contribution portfolios. This notebook introduces **Hierarchical Risk Parity**, or HRP.

HRP is useful because it:

1. uses correlation structure to cluster related assets;
2. avoids direct covariance matrix inversion;
3. allocates recursively across clusters;
4. tends to produce diversified, stable weights;
5. is often more robust than naive mean-variance optimisation when covariance estimates are noisy.

It covers:

1. correlation distance;
2. hierarchical clustering;
3. dendrogram interpretation;
4. quasi-diagonalisation;
5. inverse-variance cluster portfolios;
6. recursive bisection;
7. HRP weight construction;
8. comparison with equal weight, inverse volatility, GMV, and ERC;
9. train/test validation;
10. rolling HRP allocation;
11. turnover and transaction costs;
12. cluster-level risk contributions;
13. sensitivity to covariance noise;
14. stress-regime behaviour;
15. tail-risk comparison;
16. governance checks;
17. saved outputs and manifest.

The key idea:

> HRP allocates risk by respecting the hierarchy of correlations, rather than pretending a noisy covariance inverse is a stable object.

## 1. Why HRP?

Mean-variance optimisation often requires:

$$
\Sigma^{-1}
$$

where $\Sigma$ is the covariance matrix.

If $\Sigma$ is noisy or ill-conditioned, the optimiser can produce unstable extreme weights.

Risk parity reduces dependence on expected returns, but still depends on covariance and nonlinear optimisation.

HRP takes a different route:

1. convert correlations into distances;
2. cluster similar assets;
3. reorder the covariance matrix by cluster structure;
4. allocate capital recursively across clusters by risk.

This creates a portfolio that is aware of asset similarity and avoids direct covariance inversion.

## 2. Correlation distance

Given asset correlation:

$$
\rho_{ij}
$$

define distance:

$$
d_{ij} = \sqrt{\frac{1-\rho_{ij}}{2}}
$$

Properties:

- if $\rho_{ij}=1$, then $d_{ij}=0$;
- if $\rho_{ij}=0$, then $d_{ij}\approx0.707$;
- if $\rho_{ij}=-1$, then $d_{ij}=1$.

This transforms correlation into a distance-like quantity suitable for clustering.

## 3. HRP algorithm

A standard HRP workflow:

### Step 1 — Tree clustering

Use hierarchical clustering on correlation distance.

### Step 2 — Quasi-diagonalisation

Reorder assets so similar assets sit near each other.

### Step 3 — Recursive bisection

Split ordered assets into left and right clusters.

For each cluster, compute inverse-variance portfolio variance.

Allocate between clusters by inverse variance:

$$
\alpha = 1 - \frac{\sigma^2_{left}} {\sigma^2_{left}+\sigma^2_{right}}
$$

Then:

$$
w_{left} \leftarrow \alpha w_{left}
$$

$$
w_{right} \leftarrow (1-\alpha) w_{right}
$$

Repeat until each cluster contains one asset.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list
    from scipy.spatial.distance import squareform
    SCIPY_CLUSTER_AVAILABLE = True
except Exception:
    SCIPY_CLUSTER_AVAILABLE = False

try:
    from scipy.optimize import minimize
    SCIPY_OPTIMIZE_AVAILABLE = True
except Exception:
    SCIPY_OPTIMIZE_AVAILABLE = False

try:
    from sklearn.covariance import LedoitWolf
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

SCIPY_CLUSTER_AVAILABLE, SCIPY_OPTIMIZE_AVAILABLE, SKLEARN_AVAILABLE

In [ ]:
@dataclass(frozen=True)
class HRPConfig:
    n_days: int = 2_400
    annualisation: int = 252
    train_fraction: float = 0.65
    estimation_window: int = 252
    rebalance_step: int = 21
    transaction_cost_bps: float = 3.0
    max_weight: float = 0.40
    n_noise_trials: int = 100
    seed: int = 42


config = HRPConfig()
config

## 5. Synthetic clustered asset universe

We simulate assets with cluster structure:

| Cluster | Assets |
|---|---|
| Equity | US, Europe, Japan, EM equities |
| Bonds | US and Europe bonds |
| Commodities | Gold, oil, copper |
| Alternatives | FX carry, trend-following |
| Real assets / crypto | REIT, crypto proxy |

The true correlation structure has clusters, but stress regimes make correlations unstable.

In [ ]:
def simulate_clustered_universe(config: HRPConfig):
    rng = np.random.default_rng(config.seed)
    dates = pd.bdate_range("2014-01-01", periods=config.n_days)

    assets = [
        "US_EQ", "EU_EQ", "JP_EQ", "EM_EQ",
        "US_BOND", "EU_BOND",
        "GOLD", "OIL", "COPPER",
        "FX_CARRY", "TREND",
        "REIT", "BTC_PROXY"
    ]

    asset_class = {
        "US_EQ": "equity",
        "EU_EQ": "equity",
        "JP_EQ": "equity",
        "EM_EQ": "equity",
        "US_BOND": "bond",
        "EU_BOND": "bond",
        "GOLD": "commodity",
        "OIL": "commodity",
        "COPPER": "commodity",
        "FX_CARRY": "fx",
        "TREND": "alternative",
        "REIT": "real_asset",
        "BTC_PROXY": "crypto",
    }

    factor_names = ["equity", "rates", "commodity", "carry", "trend", "crypto", "global"]
    factor_vol_ann = np.array([0.14, 0.055, 0.13, 0.08, 0.09, 0.36, 0.10])
    factor_vol_daily = factor_vol_ann / np.sqrt(config.annualisation)

    factor_corr = np.array([
        [ 1.00, -0.25,  0.20,  0.15, -0.10,  0.35,  0.55],
        [-0.25,  1.00,  0.10, -0.05,  0.15, -0.20, -0.15],
        [ 0.20,  0.10,  1.00,  0.10,  0.05,  0.20,  0.25],
        [ 0.15, -0.05,  0.10,  1.00,  0.00,  0.15,  0.10],
        [-0.10,  0.15,  0.05,  0.00,  1.00,  0.00, -0.05],
        [ 0.35, -0.20,  0.20,  0.15,  0.00,  1.00,  0.35],
        [ 0.55, -0.15,  0.25,  0.10, -0.05,  0.35,  1.00],
    ])

    factor_cov = np.outer(factor_vol_daily, factor_vol_daily) * factor_corr

    loadings = pd.DataFrame(0.0, index=assets, columns=factor_names)
    loadings.loc[["US_EQ", "EU_EQ", "JP_EQ", "EM_EQ", "REIT"], "equity"] = [1.00, 0.95, 0.80, 1.15, 0.55]
    loadings.loc[["US_EQ", "EU_EQ", "JP_EQ", "EM_EQ", "REIT"], "global"] = [0.35, 0.35, 0.30, 0.40, 0.30]
    loadings.loc[["US_BOND", "EU_BOND"], "rates"] = [1.00, 0.90]
    loadings.loc[["US_BOND", "EU_BOND"], "global"] = [-0.15, -0.10]
    loadings.loc[["GOLD", "OIL", "COPPER"], "commodity"] = [0.45, 1.10, 0.95]
    loadings.loc[["GOLD", "OIL", "COPPER"], "global"] = [0.05, 0.20, 0.25]
    loadings.loc["FX_CARRY", "carry"] = 1.00
    loadings.loc["FX_CARRY", "global"] = 0.15
    loadings.loc["TREND", "trend"] = 1.00
    loadings.loc["TREND", "global"] = -0.20
    loadings.loc["BTC_PROXY", "crypto"] = 1.00
    loadings.loc["BTC_PROXY", "global"] = 0.45

    idio_vol_ann = pd.Series({
        "US_EQ": 0.07,
        "EU_EQ": 0.08,
        "JP_EQ": 0.09,
        "EM_EQ": 0.13,
        "US_BOND": 0.025,
        "EU_BOND": 0.030,
        "GOLD": 0.12,
        "OIL": 0.22,
        "COPPER": 0.18,
        "FX_CARRY": 0.07,
        "TREND": 0.08,
        "REIT": 0.11,
        "BTC_PROXY": 0.50,
    })

    annual_mean = pd.Series({
        "US_EQ": 0.07,
        "EU_EQ": 0.06,
        "JP_EQ": 0.045,
        "EM_EQ": 0.08,
        "US_BOND": 0.025,
        "EU_BOND": 0.020,
        "GOLD": 0.035,
        "OIL": 0.045,
        "COPPER": 0.040,
        "FX_CARRY": 0.030,
        "TREND": 0.050,
        "REIT": 0.060,
        "BTC_PROXY": 0.120,
    })

    regime = np.zeros(config.n_days, dtype=int)
    transition = np.array([[0.985, 0.015], [0.055, 0.945]])

    factor_returns = np.zeros((config.n_days, len(factor_names)))
    asset_returns = np.zeros((config.n_days, len(assets)))
    stress_type = np.array(["normal"] * config.n_days, dtype=object)

    for t in range(config.n_days):
        if t > 0:
            regime[t] = rng.choice([0, 1], p=transition[regime[t - 1]])

        vol_multiplier = 1.0 if regime[t] == 0 else 2.4
        cov_t = factor_cov * vol_multiplier**2

        z = rng.standard_t(df=6, size=len(factor_names)) * np.sqrt((6 - 2) / 6)
        L = np.linalg.cholesky(cov_t + 1e-12 * np.eye(len(factor_names)))
        f = L @ z

        u = rng.uniform()
        if u < 0.008:
            stress_type[t] = "equity_crash"
            f[0] -= rng.uniform(0.035, 0.100)
            f[1] += rng.uniform(0.004, 0.025)
            f[5] -= rng.uniform(0.080, 0.220)
            f[6] -= rng.uniform(0.020, 0.060)
        elif u < 0.014:
            stress_type[t] = "inflation_shock"
            f[1] -= rng.uniform(0.010, 0.035)
            f[2] += rng.uniform(0.030, 0.090)
            f[0] -= rng.uniform(0.010, 0.050)
        elif u < 0.020:
            stress_type[t] = "commodity_crash"
            f[2] -= rng.uniform(0.040, 0.120)
            f[6] -= rng.uniform(0.005, 0.030)
        elif u < 0.025:
            stress_type[t] = "crypto_crash"
            f[5] -= rng.uniform(0.120, 0.300)
            f[6] -= rng.uniform(0.005, 0.030)

        eps = rng.standard_t(df=5, size=len(assets)) * np.sqrt((5 - 2) / 5)
        eps = eps * (idio_vol_ann.values / np.sqrt(config.annualisation))

        mu_daily = annual_mean.values / config.annualisation
        asset_returns[t] = mu_daily + loadings.to_numpy() @ f + eps
        factor_returns[t] = f

    returns_df = pd.DataFrame(asset_returns, columns=assets)
    returns_df.insert(0, "date", dates)
    returns_df["regime"] = regime
    returns_df["stress_type"] = stress_type

    factor_df = pd.DataFrame(factor_returns, columns=factor_names)
    factor_df.insert(0, "date", dates)
    factor_df["regime"] = regime
    factor_df["stress_type"] = stress_type

    metadata = pd.DataFrame({
        "asset": assets,
        "asset_class": [asset_class[a] for a in assets],
        "idio_vol_ann": [idio_vol_ann[a] for a in assets],
        "annual_mean_assumption": [annual_mean[a] for a in assets],
    })

    return returns_df, factor_df, metadata


returns_df, factor_returns, asset_metadata = simulate_clustered_universe(config)
asset_cols = asset_metadata["asset"].tolist()
returns = returns_df.set_index("date")[asset_cols]

returns_df.head(), asset_metadata

In [ ]:
plt.figure(figsize=(12, 6))
for asset in asset_cols:
    plt.plot(returns.index, (1 + returns[asset]).cumprod(), label=asset, alpha=0.75)
plt.title("Synthetic Asset Growth of 1")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend(ncol=3)
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(returns_df["date"], returns_df["regime"], drawstyle="steps-post")
plt.title("Volatility Regime")
plt.xlabel("Date")
plt.ylabel("Regime")
plt.yticks([0, 1])
plt.show()

## 6. Train/test split and covariance estimation

We estimate covariance using the train sample.

When available, we use Ledoit-Wolf shrinkage.

This is not required by HRP, but it improves covariance robustness.

In [ ]:
split_idx = int(len(returns) * config.train_fraction)
train_returns = returns.iloc[:split_idx]
test_returns = returns.iloc[split_idx:]

def estimate_covariance(returns_window):
    X = returns_window.dropna().to_numpy()

    if SKLEARN_AVAILABLE:
        estimator = LedoitWolf().fit(X)
        cov = estimator.covariance_
        method = "sklearn_ledoit_wolf"
        shrinkage = float(estimator.shrinkage_)
    else:
        sample = np.cov(X, rowvar=False, ddof=1)
        target = np.diag(np.diag(sample))
        shrinkage = 0.30
        cov = shrinkage * target + (1 - shrinkage) * sample
        method = "fallback_diagonal_shrinkage"

    return pd.DataFrame(cov, index=returns_window.columns, columns=returns_window.columns), method, shrinkage

cov_train, cov_method, cov_shrinkage = estimate_covariance(train_returns)
corr_train = train_returns.corr()

split_metadata = {
    "n_total_days": int(len(returns)),
    "n_train_days": int(len(train_returns)),
    "n_test_days": int(len(test_returns)),
    "train_start": str(train_returns.index[0]),
    "train_end": str(train_returns.index[-1]),
    "test_start": str(test_returns.index[0]),
    "test_end": str(test_returns.index[-1]),
    "covariance_method": cov_method,
    "covariance_shrinkage": cov_shrinkage,
}

pd.Series(split_metadata)

## 7. Correlation distance matrix

Transform correlation into distance:

$$
d_{ij}=\sqrt{\frac{1-\rho_{ij}}{2}}
$$

This is the input to hierarchical clustering.

In [ ]:
def correlation_distance(corr):
    dist = np.sqrt(np.maximum(0.0, (1.0 - corr) / 2.0))
    np.fill_diagonal(dist.values, 0.0)
    return dist

dist_train = correlation_distance(corr_train)

dist_train.head()

In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(dist_train.values, vmin=0, vmax=1)
plt.colorbar(label="Correlation distance")
plt.xticks(range(len(asset_cols)), asset_cols, rotation=90)
plt.yticks(range(len(asset_cols)), asset_cols)
plt.title("Correlation Distance Matrix")
plt.tight_layout()
plt.show()

## 8. Hierarchical clustering

We use hierarchical clustering on the distance matrix.

Common linkage methods:

- single linkage;
- complete linkage;
- average linkage;
- Ward linkage.

HRP is commonly demonstrated with single linkage, but average linkage can be more stable.

This notebook uses average linkage by default.

In [ ]:
def compute_linkage(distance_df, method="average"):
    if SCIPY_CLUSTER_AVAILABLE:
        condensed = squareform(distance_df.values, checks=False)
        return linkage(condensed, method=method)

    # Fallback: no linkage matrix. The fallback order uses a simple greedy nearest-neighbour path.
    return None


linkage_matrix = compute_linkage(dist_train, method="average")

linkage_matrix[:5] if linkage_matrix is not None else "SciPy clustering unavailable; using fallback order."

In [ ]:
if SCIPY_CLUSTER_AVAILABLE and linkage_matrix is not None:
    plt.figure(figsize=(12, 6))
    dendrogram(linkage_matrix, labels=asset_cols, leaf_rotation=90)
    plt.title("Hierarchical Clustering Dendrogram")
    plt.ylabel("Distance")
    plt.tight_layout()
    plt.show()
else:
    print("Dendrogram skipped because scipy clustering is unavailable.")

## 9. Quasi-diagonalisation

Quasi-diagonalisation reorders assets so similar assets are near each other.

This does not change the covariance matrix mathematically.

It makes cluster structure visible and prepares recursive bisection.

In [ ]:
def fallback_leaf_order(distance_df):
    remaining = list(distance_df.index)
    if not remaining:
        return []

    order = [remaining.pop(0)]

    while remaining:
        last = order[-1]
        next_asset = min(remaining, key=lambda x: distance_df.loc[last, x])
        order.append(next_asset)
        remaining.remove(next_asset)

    return order


def get_quasi_diag_order(linkage_matrix, labels, distance_df):
    if SCIPY_CLUSTER_AVAILABLE and linkage_matrix is not None:
        leaf_ids = leaves_list(linkage_matrix)
        return [labels[i] for i in leaf_ids]

    return fallback_leaf_order(distance_df)


ordered_assets = get_quasi_diag_order(linkage_matrix, asset_cols, dist_train)

ordered_assets

In [ ]:
corr_ordered = corr_train.loc[ordered_assets, ordered_assets]

plt.figure(figsize=(10, 8))
plt.imshow(corr_ordered.values, vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(ordered_assets)), ordered_assets, rotation=90)
plt.yticks(range(len(ordered_assets)), ordered_assets)
plt.title("Quasi-Diagonalised Correlation Matrix")
plt.tight_layout()
plt.show()

## 10. Cluster variance

Within a cluster, HRP often uses inverse-variance weights:

$$
w_i^{IVP} = \frac{1/\sigma_i^2} {\sum_{j\in C}1/\sigma_j^2}
$$

Cluster variance:

$$
\sigma_C^2 = w_C^\top \Sigma_C w_C
$$

This gives a risk estimate for the whole cluster.

In [ ]:
def inverse_variance_portfolio(cov):
    diag = np.diag(cov)
    inv_diag = 1.0 / np.maximum(diag, 1e-12)
    weights = inv_diag / inv_diag.sum()
    return weights


def cluster_variance(cov_df, cluster_assets):
    sub_cov = cov_df.loc[cluster_assets, cluster_assets].to_numpy()
    w = inverse_variance_portfolio(sub_cov)
    variance = float(w.T @ sub_cov @ w)
    return variance


cluster_variance(cov_train, ordered_assets[:4])

## 11. HRP recursive bisection

Recursive bisection:

1. start with all ordered assets;
2. split into left and right halves;
3. compute cluster variance for each half;
4. allocate capital inversely to cluster variance;
5. repeat until each cluster contains one asset.

This creates HRP weights.

In [ ]:
def recursive_bisection_hrp(cov_df, ordered_assets):
    weights = pd.Series(1.0, index=ordered_assets)

    clusters = [ordered_assets]

    while clusters:
        new_clusters = []

        for cluster in clusters:
            if len(cluster) <= 1:
                continue

            split = len(cluster) // 2
            left = cluster[:split]
            right = cluster[split:]

            var_left = cluster_variance(cov_df, left)
            var_right = cluster_variance(cov_df, right)

            alpha = 1.0 - var_left / (var_left + var_right) if (var_left + var_right) > 0 else 0.5

            weights[left] *= alpha
            weights[right] *= (1.0 - alpha)

            new_clusters.append(left)
            new_clusters.append(right)

        clusters = new_clusters

    weights = weights / weights.sum()
    return weights.reindex(cov_df.index).fillna(0.0)


hrp_weights = recursive_bisection_hrp(cov_train, ordered_assets)

hrp_weights.to_frame("hrp_weight").sort_values("hrp_weight", ascending=False)

## 12. Comparator portfolios

We compare HRP to:

1. equal weight;
2. inverse volatility;
3. global minimum variance;
4. equal risk contribution.

This shows where HRP sits between simple rules and nonlinear optimisation.

In [ ]:
def portfolio_volatility(weights, cov):
    w = np.asarray(weights, dtype=float)
    var = float(w.T @ cov @ w)
    return np.sqrt(max(var, 0.0))


def inverse_vol_weights(cov_df):
    diag = np.diag(cov_df.to_numpy())
    inv_vol = 1.0 / np.sqrt(np.maximum(diag, 1e-12))
    return pd.Series(inv_vol / inv_vol.sum(), index=cov_df.index)


def normalise_weights(w, min_weight=0.0, max_weight=1.0):
    w = np.asarray(w, dtype=float)
    w = np.clip(w, min_weight, max_weight)
    if w.sum() <= 0:
        w = np.ones_like(w) / len(w)
    w = w / w.sum()
    return w


def gmv_weights(cov_df):
    cov = cov_df.to_numpy()
    n = cov.shape[0]
    x0 = np.ones(n) / n

    def obj(w):
        return float(w.T @ cov @ w)

    if SCIPY_OPTIMIZE_AVAILABLE:
        res = minimize(
            obj,
            x0=x0,
            method="SLSQP",
            bounds=[(0.0, config.max_weight)] * n,
            constraints=[{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}],
            options={"maxiter": 1000, "ftol": 1e-12}
        )
        if res.success:
            return pd.Series(normalise_weights(res.x, 0.0, config.max_weight), index=cov_df.index)

    inv_cov = np.linalg.pinv(cov)
    raw = inv_cov @ np.ones(n)
    raw = raw / raw.sum()
    return pd.Series(normalise_weights(raw, 0.0, config.max_weight), index=cov_df.index)


def risk_contributions(weights, cov_df):
    w = np.asarray(weights, dtype=float)
    cov = cov_df.to_numpy()
    sigma = portfolio_volatility(w, cov)

    if sigma <= 0:
        pct = np.zeros_like(w)
        rc = np.zeros_like(w)
        mrc = np.zeros_like(w)
    else:
        mrc = cov @ w / sigma
        rc = w * mrc
        pct = rc / sigma

    return {
        "portfolio_vol": sigma,
        "marginal_risk_contribution": mrc,
        "component_risk_contribution": rc,
        "pct_risk_contribution": pct
    }


def erc_objective(w, cov_df):
    n = len(w)
    target = np.ones(n) / n
    pct = risk_contributions(w, cov_df)["pct_risk_contribution"]
    return float(np.sum((pct - target) ** 2))


def erc_weights(cov_df):
    n = cov_df.shape[0]
    x0 = inverse_vol_weights(cov_df).to_numpy()

    if SCIPY_OPTIMIZE_AVAILABLE:
        res = minimize(
            lambda w: erc_objective(w, cov_df),
            x0=x0,
            method="SLSQP",
            bounds=[(0.0, config.max_weight)] * n,
            constraints=[{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}],
            options={"maxiter": 2000, "ftol": 1e-12}
        )
        if res.success:
            return pd.Series(normalise_weights(res.x, 0.0, config.max_weight), index=cov_df.index), {
                "success": True,
                "method": "scipy_slsqp",
                "objective": float(res.fun)
            }

    rng = np.random.default_rng(config.seed)
    best_w = x0
    best_obj = erc_objective(best_w, cov_df)

    for _ in range(20_000):
        w = rng.dirichlet(np.ones(n))
        w = normalise_weights(w, 0.0, config.max_weight)
        val = erc_objective(w, cov_df)
        if val < best_obj:
            best_w = w
            best_obj = val

    return pd.Series(best_w, index=cov_df.index), {
        "success": False,
        "method": "fallback_random_search",
        "objective": float(best_obj)
    }


equal_weights = pd.Series(1.0 / len(asset_cols), index=asset_cols)
inverse_vol = inverse_vol_weights(cov_train)
gmv = gmv_weights(cov_train)
erc, erc_info = erc_weights(cov_train)

static_weights = pd.DataFrame({
    "equal_weight": equal_weights,
    "inverse_vol": inverse_vol,
    "global_min_variance": gmv,
    "equal_risk_contribution": erc,
    "hierarchical_risk_parity": hrp_weights,
})

static_weights

In [ ]:
plt.figure(figsize=(12, 7))
bottom = np.zeros(static_weights.shape[1])
x = np.arange(static_weights.shape[1])

for asset in static_weights.index:
    plt.bar(x, static_weights.loc[asset], bottom=bottom, label=asset)
    bottom += static_weights.loc[asset].values

plt.xticks(x, static_weights.columns, rotation=45, ha="right")
plt.title("Static Portfolio Weights")
plt.ylabel("Weight")
plt.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

## 13. Static risk contribution comparison

HRP is not designed to equalise every individual risk contribution exactly.

Instead, it diversifies recursively across clusters.

We compare its risk contributions to equal weight, inverse vol, GMV, and ERC.

In [ ]:
def risk_contribution_table(weights_df, cov_df):
    rows = []

    for portfolio in weights_df.columns:
        w = weights_df[portfolio].to_numpy()
        rc = risk_contributions(w, cov_df)

        for asset, weight, pct in zip(weights_df.index, w, rc["pct_risk_contribution"]):
            rows.append({
                "portfolio": portfolio,
                "asset": asset,
                "weight": weight,
                "pct_risk_contribution": pct,
                "asset_class": asset_metadata.set_index("asset").loc[asset, "asset_class"],
            })

    return pd.DataFrame(rows)


static_rc = risk_contribution_table(static_weights, cov_train)
static_rc.head()

In [ ]:
for portfolio in static_weights.columns:
    sub = static_rc[static_rc["portfolio"] == portfolio].sort_values("pct_risk_contribution")

    plt.figure(figsize=(10, 6))
    plt.barh(sub["asset"], sub["pct_risk_contribution"])
    plt.axvline(1 / len(asset_cols), linestyle="--", label="equal asset risk")
    plt.title(f"Risk Contributions: {portfolio}")
    plt.xlabel("Percent risk contribution")
    plt.ylabel("Asset")
    plt.legend()
    plt.show()

## 14. Static portfolio diagnostics

Diagnostics:

- expected volatility;
- effective number of capital bets;
- effective number of risk bets;
- max weight;
- cluster concentration.

In [ ]:
def effective_number(x):
    x = np.asarray(x, dtype=float)
    x = np.abs(x)
    total = x.sum()
    if total <= 0:
        return np.nan
    p = x / total
    return float(1.0 / np.sum(p**2))


def static_portfolio_diagnostics(weights_df, cov_df):
    rows = []

    for portfolio in weights_df.columns:
        w = weights_df[portfolio].to_numpy()
        rc = risk_contributions(w, cov_df)["pct_risk_contribution"]

        rows.append({
            "portfolio": portfolio,
            "expected_vol_ann": portfolio_volatility(w, cov_df.to_numpy()) * np.sqrt(config.annualisation),
            "max_weight": float(np.max(w)),
            "effective_n_capital": effective_number(w),
            "max_pct_risk_contribution": float(np.max(rc)),
            "effective_n_risk": effective_number(rc),
        })

    return pd.DataFrame(rows).sort_values("effective_n_risk", ascending=False)


static_diagnostics = static_portfolio_diagnostics(static_weights, cov_train)
static_diagnostics

## 15. Static out-of-sample backtest

Estimate weights on train data and apply them to test data.

This tests whether in-sample diversification survives out of sample.

In [ ]:
def max_drawdown(nav):
    dd = nav / nav.cummax() - 1.0
    return dd, float(dd.min())


def backtest_static_weights(test_returns, weights_df, cost_bps):
    out = pd.DataFrame(index=test_returns.index)
    rows = []

    for portfolio in weights_df.columns:
        w = weights_df[portfolio]
        gross = test_returns @ w

        cost = pd.Series(0.0, index=test_returns.index)
        cost.iloc[0] = w.abs().sum() * cost_bps / 10000.0

        net = gross - cost
        nav = (1 + net).cumprod()
        dd, mdd = max_drawdown(nav)

        out[f"{portfolio}_return"] = net
        out[f"{portfolio}_nav"] = nav

        rows.append({
            "portfolio": portfolio,
            "annualised_return": float(net.mean() * config.annualisation),
            "annualised_vol": float(net.std(ddof=1) * np.sqrt(config.annualisation)),
            "sharpe_proxy": float(net.mean() / net.std(ddof=1) * np.sqrt(config.annualisation)) if net.std(ddof=1) > 0 else np.nan,
            "max_drawdown": mdd,
            "worst_day": float(net.min()),
            "total_return": float(nav.iloc[-1] - 1.0),
            "initial_cost": float(cost.iloc[0]),
        })

    return out, pd.DataFrame(rows).sort_values("sharpe_proxy", ascending=False)


static_backtest, static_performance = backtest_static_weights(test_returns, static_weights, config.transaction_cost_bps)

static_performance

In [ ]:
plt.figure(figsize=(12, 6))
for portfolio in static_weights.columns:
    plt.plot(static_backtest.index, static_backtest[f"{portfolio}_nav"], label=portfolio)
plt.title("Static Out-of-Sample Backtest")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend()
plt.show()

## 16. Rolling HRP allocation

A practical HRP portfolio is re-estimated periodically.

At each rebalance:

1. estimate covariance and correlation from trailing window;
2. compute correlation distance;
3. cluster assets;
4. quasi-diagonalise;
5. recursively allocate HRP weights;
6. hold until next rebalance.

We compare rolling HRP with rolling equal weight, inverse vol, GMV, and ERC.

In [ ]:
def hrp_weights_from_returns(returns_window):
    cov_df, method, shrinkage = estimate_covariance(returns_window)
    corr_df = returns_window.corr()
    dist_df = correlation_distance(corr_df)
    link = compute_linkage(dist_df, method="average")
    order = get_quasi_diag_order(link, list(returns_window.columns), dist_df)
    weights = recursive_bisection_hrp(cov_df, order)
    return weights, {
        "cov_method": method,
        "shrinkage": shrinkage,
        "ordered_assets": "|".join(order),
        "linkage_available": SCIPY_CLUSTER_AVAILABLE,
    }


def rolling_allocation(returns, method):
    rows = []
    diagnostics = []

    current = config.estimation_window

    while current < len(returns):
        window = returns.iloc[current - config.estimation_window:current]
        cov_df, cov_method, shrinkage = estimate_covariance(window)

        if method == "equal_weight":
            w = pd.Series(1.0 / len(asset_cols), index=asset_cols)
            info = {"cov_method": cov_method, "shrinkage": shrinkage, "ordered_assets": "", "linkage_available": SCIPY_CLUSTER_AVAILABLE}
        elif method == "inverse_vol":
            w = inverse_vol_weights(cov_df)
            info = {"cov_method": cov_method, "shrinkage": shrinkage, "ordered_assets": "", "linkage_available": SCIPY_CLUSTER_AVAILABLE}
        elif method == "global_min_variance":
            w = gmv_weights(cov_df)
            info = {"cov_method": cov_method, "shrinkage": shrinkage, "ordered_assets": "", "linkage_available": SCIPY_CLUSTER_AVAILABLE}
        elif method == "equal_risk_contribution":
            w, erc_info_local = erc_weights(cov_df)
            info = {"cov_method": cov_method, "shrinkage": shrinkage, "ordered_assets": "", "linkage_available": SCIPY_CLUSTER_AVAILABLE, "erc_method": erc_info_local["method"]}
        elif method == "hierarchical_risk_parity":
            w, info = hrp_weights_from_returns(window)
        else:
            raise ValueError("Unknown method.")

        hold_end = min(current + config.rebalance_step, len(returns))

        for date in returns.index[current:hold_end]:
            rows.append(pd.Series(w, name=date))

        rc = risk_contributions(w.to_numpy(), cov_df)["pct_risk_contribution"]

        diagnostics.append({
            "rebalance_date": returns.index[current],
            "method": method,
            "cov_method": info.get("cov_method"),
            "shrinkage": info.get("shrinkage"),
            "ordered_assets": info.get("ordered_assets", ""),
            "expected_vol_ann": portfolio_volatility(w.to_numpy(), cov_df.to_numpy()) * np.sqrt(config.annualisation),
            "effective_n_capital": effective_number(w.to_numpy()),
            "effective_n_risk": effective_number(rc),
            "max_weight": float(w.max()),
            "max_pct_risk_contribution": float(np.max(rc)),
        })

        current += config.rebalance_step

    weights = pd.DataFrame(rows)
    weights.index.name = "date"
    weights = weights.reindex(returns.index).ffill().fillna(0.0)

    return weights, pd.DataFrame(diagnostics)


methods = [
    "equal_weight",
    "inverse_vol",
    "global_min_variance",
    "equal_risk_contribution",
    "hierarchical_risk_parity",
]

rolling_weights = {}
rolling_diagnostics = {}

for method in methods:
    weights, diag = rolling_allocation(returns, method)
    rolling_weights[method] = weights
    rolling_diagnostics[method] = diag

rolling_diagnostics["hierarchical_risk_parity"].head()

## 17. Rolling backtest with costs

Weights at $t-1$ are applied to return at $t$.

Transaction cost:

$$
cost_t = c\sum_i |w_{i,t}-w_{i,t-1}|
$$

In [ ]:
def backtest_dynamic_weights(returns, weights, cost_bps):
    investable = weights.shift(1).fillna(0.0)
    gross = (investable * returns).sum(axis=1)

    turnover = weights.diff().abs().sum(axis=1).fillna(weights.abs().sum(axis=1))
    cost = turnover * cost_bps / 10000.0

    net = gross - cost
    nav = (1 + net).cumprod()

    return pd.DataFrame({
        "gross_return": gross,
        "transaction_cost": cost,
        "net_return": net,
        "nav": nav,
        "turnover": turnover,
        "gross_exposure": investable.abs().sum(axis=1),
    })


rolling_backtests = {
    method: backtest_dynamic_weights(returns, weights, config.transaction_cost_bps)
    for method, weights in rolling_weights.items()
}


def summarise_dynamic_backtests(backtests):
    rows = []

    for method, bt in backtests.items():
        sub = bt.iloc[config.estimation_window:].copy()
        r = sub["net_return"]
        nav = (1 + r).cumprod()
        dd, mdd = max_drawdown(nav)

        rows.append({
            "portfolio": method,
            "annualised_return": float(r.mean() * config.annualisation),
            "annualised_vol": float(r.std(ddof=1) * np.sqrt(config.annualisation)),
            "sharpe_proxy": float(r.mean() / r.std(ddof=1) * np.sqrt(config.annualisation)) if r.std(ddof=1) > 0 else np.nan,
            "max_drawdown": mdd,
            "worst_day": float(r.min()),
            "total_return": float(nav.iloc[-1] - 1.0),
            "mean_turnover": float(sub["turnover"].mean()),
            "annualised_cost_drag": float(sub["transaction_cost"].mean() * config.annualisation),
            "mean_gross_exposure": float(sub["gross_exposure"].mean()),
        })

    return pd.DataFrame(rows).sort_values("sharpe_proxy", ascending=False)


rolling_performance = summarise_dynamic_backtests(rolling_backtests)
rolling_performance

In [ ]:
plt.figure(figsize=(12, 6))
for method, bt in rolling_backtests.items():
    sub = bt.iloc[config.estimation_window:]
    nav = (1 + sub["net_return"]).cumprod()
    plt.plot(sub.index, nav, label=method)
plt.title("Rolling Allocation Backtest")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend()
plt.show()

plt.figure(figsize=(12, 5))
for method in ["inverse_vol", "equal_risk_contribution", "hierarchical_risk_parity"]:
    plt.plot(rolling_backtests[method].index, rolling_backtests[method]["turnover"].rolling(21).mean(), label=method)
plt.title("21-Day Average Turnover")
plt.xlabel("Date")
plt.ylabel("Turnover")
plt.legend()
plt.show()

## 18. Rolling HRP weights

Inspect HRP allocation over time.

A good HRP process should be stable but still respond to changing risk structure.

In [ ]:
plt.figure(figsize=(12, 6))
for asset in asset_cols:
    plt.plot(rolling_weights["hierarchical_risk_parity"].index, rolling_weights["hierarchical_risk_parity"][asset], label=asset, alpha=0.75)
plt.title("Rolling HRP Weights")
plt.xlabel("Date")
plt.ylabel("Weight")
plt.legend(ncol=3)
plt.show()

## 19. Realised risk contributions

We compute realised risk contributions using the covariance of the out-of-sample period and average weights.

In [ ]:
realised_cov = returns.iloc[config.estimation_window:].cov()

realised_rc_rows = []

for method, weights in rolling_weights.items():
    avg_w = weights.iloc[config.estimation_window:].mean()
    rc = risk_contributions(avg_w.to_numpy(), realised_cov)["pct_risk_contribution"]

    for asset, weight, pct in zip(asset_cols, avg_w, rc):
        realised_rc_rows.append({
            "portfolio": method,
            "asset": asset,
            "average_weight": weight,
            "realised_pct_risk_contribution": pct,
            "asset_class": asset_metadata.set_index("asset").loc[asset, "asset_class"],
        })

realised_rc = pd.DataFrame(realised_rc_rows)

realised_rc.head()

In [ ]:
for method in ["equal_weight", "equal_risk_contribution", "hierarchical_risk_parity"]:
    sub = realised_rc[realised_rc["portfolio"] == method].sort_values("realised_pct_risk_contribution")

    plt.figure(figsize=(10, 6))
    plt.barh(sub["asset"], sub["realised_pct_risk_contribution"])
    plt.axvline(1 / len(asset_cols), linestyle="--", label="equal asset risk")
    plt.title(f"Realised Risk Contributions: {method}")
    plt.xlabel("Percent risk contribution")
    plt.ylabel("Asset")
    plt.legend()
    plt.show()

## 20. Cluster-level risk contributions

HRP is cluster-aware.

We aggregate realised risk contributions by asset class as a proxy for cluster-level risk.

In [ ]:
class_rc = (
    realised_rc
    .groupby(["portfolio", "asset_class"])
    .agg(
        realised_pct_risk_contribution=("realised_pct_risk_contribution", "sum"),
        average_weight=("average_weight", "sum")
    )
    .reset_index()
)

class_rc.sort_values(["portfolio", "realised_pct_risk_contribution"], ascending=[True, False]).head(20)

In [ ]:
plot_class = class_rc.pivot(index="asset_class", columns="portfolio", values="realised_pct_risk_contribution").fillna(0.0)

plt.figure(figsize=(12, 6))
x = np.arange(len(plot_class.index))
width = 0.16

for i, portfolio in enumerate(plot_class.columns):
    plt.bar(x + (i - len(plot_class.columns)/2) * width, plot_class[portfolio], width=width, label=portfolio)

plt.xticks(x, plot_class.index, rotation=45, ha="right")
plt.title("Realised Asset-Class Risk Contributions")
plt.ylabel("Risk contribution")
plt.legend(ncol=2)
plt.tight_layout()
plt.show()

## 21. Sensitivity to covariance noise

A key claim of HRP is improved robustness.

We perturb the training returns with small noise, recompute weights, and measure weight instability.

If a method's weights change violently under tiny noise, it is fragile.

In [ ]:
def compute_static_weights_from_window(returns_window, method, seed_offset=0):
    cov_df, _, _ = estimate_covariance(returns_window)

    if method == "equal_weight":
        return pd.Series(1.0 / len(asset_cols), index=asset_cols)
    if method == "inverse_vol":
        return inverse_vol_weights(cov_df)
    if method == "global_min_variance":
        return gmv_weights(cov_df)
    if method == "equal_risk_contribution":
        w, _ = erc_weights(cov_df)
        return w
    if method == "hierarchical_risk_parity":
        w, _ = hrp_weights_from_returns(returns_window)
        return w

    raise ValueError("Unknown method.")


rng = np.random.default_rng(config.seed + 999)
noise_rows = []
weight_noise_records = []

base_window = train_returns.copy()
asset_daily_vol = base_window.std()

for trial in range(config.n_noise_trials):
    noise = pd.DataFrame(
        rng.normal(0, 0.05, size=base_window.shape),
        index=base_window.index,
        columns=base_window.columns
    ).multiply(asset_daily_vol, axis=1)

    noisy_window = base_window + noise

    for method in methods:
        w = compute_static_weights_from_window(noisy_window, method, seed_offset=trial)
        base_w = static_weights[method] if method in static_weights.columns else compute_static_weights_from_window(base_window, method)

        l1_change = float(np.abs(w - base_w).sum())

        noise_rows.append({
            "trial": trial,
            "method": method,
            "l1_weight_change_vs_base": l1_change,
            "max_abs_weight_change": float(np.max(np.abs(w - base_w))),
            "effective_n": effective_number(w),
        })

        for asset, value in w.items():
            weight_noise_records.append({
                "trial": trial,
                "method": method,
                "asset": asset,
                "weight": value
            })

noise_sensitivity = pd.DataFrame(noise_rows)
noise_weights = pd.DataFrame(weight_noise_records)

noise_summary = (
    noise_sensitivity
    .groupby("method")
    .agg(
        mean_l1_change=("l1_weight_change_vs_base", "mean"),
        p95_l1_change=("l1_weight_change_vs_base", lambda x: x.quantile(0.95)),
        mean_max_abs_change=("max_abs_weight_change", "mean"),
        mean_effective_n=("effective_n", "mean")
    )
    .reset_index()
    .sort_values("mean_l1_change")
)

noise_summary

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(noise_summary["method"], noise_summary["mean_l1_change"])
plt.title("Weight Sensitivity to Small Return Noise")
plt.xlabel("Mean L1 weight change")
plt.ylabel("Method")
plt.show()

## 22. Stress-regime behaviour

We compare performance during calm and stress regimes.

HRP can diversify clusters, but stress correlations can still rise and hurt all risky assets.

In [ ]:
regime_series = returns_df.set_index("date")["regime"].reindex(returns.index)
stress_type_series = returns_df.set_index("date")["stress_type"].reindex(returns.index)

stress_rows = []

for method, bt in rolling_backtests.items():
    sub = bt.iloc[config.estimation_window:].copy()
    sub["regime"] = regime_series.reindex(sub.index)
    sub["stress_type"] = stress_type_series.reindex(sub.index)

    for regime_value, group in sub.groupby("regime"):
        stress_rows.append({
            "portfolio": method,
            "regime": int(regime_value),
            "n_days": int(len(group)),
            "mean_return_ann": float(group["net_return"].mean() * config.annualisation),
            "vol_ann": float(group["net_return"].std(ddof=1) * np.sqrt(config.annualisation)),
            "worst_day": float(group["net_return"].min()),
            "mean_turnover": float(group["turnover"].mean()),
        })

stress_regime_report = pd.DataFrame(stress_rows)
stress_regime_report.head()

In [ ]:
plt.figure(figsize=(12, 6))
for method in ["equal_weight", "equal_risk_contribution", "hierarchical_risk_parity"]:
    sub = stress_regime_report[stress_regime_report["portfolio"] == method]
    plt.plot(sub["regime"], sub["vol_ann"], marker="o", label=method)
plt.title("Realised Volatility by Regime")
plt.xlabel("Regime")
plt.ylabel("Annualised volatility")
plt.xticks([0, 1])
plt.legend()
plt.show()

## 23. VaR and CVaR comparison

Risk clustering and diversification should also be checked in the tail.

We compare historical VaR and CVaR.

In [ ]:
def historical_var_cvar(losses, alpha):
    losses = np.asarray(losses, dtype=float)
    var = np.quantile(losses, alpha)
    tail = losses[losses >= var]
    cvar = tail.mean() if len(tail) else np.nan
    return float(var), float(cvar)


tail_rows = []

for method, bt in rolling_backtests.items():
    r = bt.iloc[config.estimation_window:]["net_return"]
    losses = -r

    for alpha in [0.95, 0.99]:
        var, cvar = historical_var_cvar(losses, alpha)
        tail_rows.append({
            "portfolio": method,
            "alpha": alpha,
            "VaR": var,
            "CVaR": cvar
        })

tail_risk_report = pd.DataFrame(tail_rows)
tail_risk_report

In [ ]:
sub = tail_risk_report[tail_risk_report["alpha"] == 0.99].sort_values("CVaR")

plt.figure(figsize=(10, 6))
plt.barh(sub["portfolio"], sub["CVaR"])
plt.title("99% Historical CVaR")
plt.xlabel("CVaR loss")
plt.ylabel("Portfolio")
plt.show()

## 24. Diversification ratio

Diversification ratio:

$$
DR = \frac{w^\top\sigma}{\sqrt{w^\top\Sigma w}}
$$

A higher value indicates more diversification benefit.

It is not the HRP objective, but it is a useful diagnostic.

In [ ]:
def diversification_ratio(weights, cov_df):
    w = np.asarray(weights, dtype=float)
    cov = cov_df.to_numpy()
    vols = np.sqrt(np.diag(cov))
    numerator = float(w @ vols)
    denominator = portfolio_volatility(w, cov)
    return numerator / denominator if denominator > 0 else np.nan


diversification_report = pd.DataFrame([
    {
        "portfolio": method,
        "diversification_ratio_train": diversification_ratio(static_weights[method], cov_train)
    }
    for method in static_weights.columns
]).sort_values("diversification_ratio_train", ascending=False)

diversification_report

## 25. Governance flags

HRP allocation should be reviewed if:

1. one asset receives too much weight;
2. one cluster dominates realised risk;
3. turnover is high;
4. tail risk is high;
5. linkage order changes violently;
6. covariance estimation fails.

In [ ]:
governance_rows = []

for method in methods:
    perf = rolling_performance[rolling_performance["portfolio"] == method].iloc[0]
    rc = realised_rc[realised_rc["portfolio"] == method]
    class_sub = class_rc[class_rc["portfolio"] == method]
    tail99 = tail_risk_report[(tail_risk_report["portfolio"] == method) & (tail_risk_report["alpha"] == 0.99)].iloc[0]

    governance_rows.append({
        "portfolio": method,
        "annualised_vol": perf["annualised_vol"],
        "max_drawdown": perf["max_drawdown"],
        "mean_turnover": perf["mean_turnover"],
        "max_weight_average": rolling_weights[method].iloc[config.estimation_window:].mean().max(),
        "max_asset_risk_contribution": rc["realised_pct_risk_contribution"].max(),
        "max_class_risk_contribution": class_sub["realised_pct_risk_contribution"].max(),
        "cvar99": tail99["CVaR"],
        "flag_weight_above_35pct": bool(rolling_weights[method].iloc[config.estimation_window:].mean().max() > 0.35),
        "flag_asset_rc_above_30pct": bool(rc["realised_pct_risk_contribution"].max() > 0.30),
        "flag_class_rc_above_55pct": bool(class_sub["realised_pct_risk_contribution"].max() > 0.55),
        "flag_turnover_above_20pct_daily_avg": bool(perf["mean_turnover"] > 0.20),
        "flag_drawdown_below_minus_20pct": bool(perf["max_drawdown"] < -0.20),
    })

governance_flags = pd.DataFrame(governance_rows)
governance_flags["review_required"] = governance_flags[
    [
        "flag_weight_above_35pct",
        "flag_asset_rc_above_30pct",
        "flag_class_rc_above_55pct",
        "flag_turnover_above_20pct_daily_avg",
        "flag_drawdown_below_minus_20pct",
    ]
].any(axis=1)

governance_flags

## 26. Audit checks

Numerical checks:

1. HRP weights sum to one;
2. HRP weights are non-negative;
3. distance matrix diagonal is zero;
4. distance matrix is symmetric;
5. covariance matrix is positive semi-definite up to tolerance;
6. backtest returns are finite.

In [ ]:
audit_rows = []

audit_rows.append({
    "check": "hrp_weights_sum_to_one",
    "value": float(abs(hrp_weights.sum() - 1.0)),
    "passed": bool(abs(hrp_weights.sum() - 1.0) < 1e-10)
})

audit_rows.append({
    "check": "hrp_weights_nonnegative",
    "value": float(hrp_weights.min()),
    "passed": bool(hrp_weights.min() >= -1e-12)
})

audit_rows.append({
    "check": "distance_diagonal_zero",
    "value": float(np.max(np.abs(np.diag(dist_train)))),
    "passed": bool(np.max(np.abs(np.diag(dist_train))) < 1e-12)
})

audit_rows.append({
    "check": "distance_matrix_symmetric",
    "value": float(np.max(np.abs(dist_train.values - dist_train.values.T))),
    "passed": bool(np.max(np.abs(dist_train.values - dist_train.values.T)) < 1e-12)
})

eigvals = np.linalg.eigvalsh(cov_train.to_numpy())
audit_rows.append({
    "check": "covariance_psd_tolerance",
    "value": float(eigvals.min()),
    "passed": bool(eigvals.min() > -1e-10)
})

finite_backtests = all(np.isfinite(bt["net_return"]).all() for bt in rolling_backtests.values())
audit_rows.append({
    "check": "rolling_backtest_returns_finite",
    "value": bool(finite_backtests),
    "passed": bool(finite_backtests)
})

audit_report = pd.DataFrame(audit_rows)
audit_report

## 27. Practical checklist for HRP

Before trusting an HRP portfolio:

1. **Inspect the dendrogram**  
   Do clusters make economic sense?

2. **Check ordered correlation matrix**  
   Does quasi-diagonalisation reveal blocks?

3. **Check weights**  
   Are weights stable and interpretable?

4. **Check cluster concentration**  
   Does one cluster dominate capital or risk?

5. **Compare to simple baselines**  
   Equal weight and inverse vol are hard to beat.

6. **Backtest out of sample**  
   In-sample clusters may not persist.

7. **Check turnover**  
   Rolling clustering can change allocations.

8. **Stress test**  
   Correlations can rise and clusters can collapse.

9. **Check tail risk**  
   HRP is not automatically CVaR-optimal.

10. **Avoid dendrogram mysticism**  
   Clustering is a tool, not a guarantee.

## 28. Saving outputs

The notebook saves:

1. synthetic returns;
2. factor returns;
3. asset metadata;
4. covariance and correlation estimates;
5. correlation distance matrix;
6. linkage matrix if available;
7. ordered assets;
8. static HRP and comparator weights;
9. static risk contributions;
10. static diagnostics;
11. static backtest;
12. rolling weights;
13. rolling diagnostics;
14. rolling backtests;
15. realised risk contributions;
16. cluster/class risk contributions;
17. noise sensitivity report;
18. stress regime report;
19. tail risk report;
20. diversification report;
21. governance flags;
22. audit report;
23. manifest.

In [ ]:
output_dir = Path("data/processed/hierarchical_risk_parity_v1")
output_dir.mkdir(parents=True, exist_ok=True)

returns_path = output_dir / "synthetic_returns.csv"
factor_returns_path = output_dir / "factor_returns.csv"
metadata_path = output_dir / "asset_metadata.csv"
cov_train_path = output_dir / "train_covariance.csv"
corr_train_path = output_dir / "train_correlation.csv"
distance_path = output_dir / "correlation_distance.csv"
ordered_assets_path = output_dir / "ordered_assets.json"
linkage_path = output_dir / "linkage_matrix.csv"
static_weights_path = output_dir / "static_weights.csv"
static_rc_path = output_dir / "static_risk_contributions.csv"
static_diagnostics_path = output_dir / "static_diagnostics.csv"
static_performance_path = output_dir / "static_performance.csv"
static_backtest_path = output_dir / "static_backtest.csv"
rolling_performance_path = output_dir / "rolling_performance.csv"
realised_rc_path = output_dir / "realised_risk_contributions.csv"
class_rc_path = output_dir / "class_risk_contributions.csv"
noise_sensitivity_path = output_dir / "noise_sensitivity.csv"
noise_summary_path = output_dir / "noise_summary.csv"
stress_regime_path = output_dir / "stress_regime_report.csv"
tail_risk_path = output_dir / "tail_risk_report.csv"
diversification_path = output_dir / "diversification_report.csv"
governance_flags_path = output_dir / "governance_flags.csv"
audit_report_path = output_dir / "audit_report.csv"
manifest_path = output_dir / "manifest.json"

returns_df.to_csv(returns_path, index=False)
factor_returns.to_csv(factor_returns_path, index=False)
asset_metadata.to_csv(metadata_path, index=False)
cov_train.to_csv(cov_train_path)
corr_train.to_csv(corr_train_path)
dist_train.to_csv(distance_path)
Path(ordered_assets_path).write_text(json.dumps({"ordered_assets": ordered_assets}, indent=2))

if linkage_matrix is not None:
    pd.DataFrame(linkage_matrix, columns=["idx1", "idx2", "distance", "count"]).to_csv(linkage_path, index=False)
else:
    pd.DataFrame().to_csv(linkage_path, index=False)

static_weights.to_csv(static_weights_path)
static_rc.to_csv(static_rc_path, index=False)
static_diagnostics.to_csv(static_diagnostics_path, index=False)
static_performance.to_csv(static_performance_path, index=False)
static_backtest.to_csv(static_backtest_path)
rolling_performance.to_csv(rolling_performance_path, index=False)
realised_rc.to_csv(realised_rc_path, index=False)
class_rc.to_csv(class_rc_path, index=False)
noise_sensitivity.to_csv(noise_sensitivity_path, index=False)
noise_summary.to_csv(noise_summary_path, index=False)
stress_regime_report.to_csv(stress_regime_path, index=False)
tail_risk_report.to_csv(tail_risk_path, index=False)
diversification_report.to_csv(diversification_path, index=False)
governance_flags.to_csv(governance_flags_path, index=False)
audit_report.to_csv(audit_report_path, index=False)

rolling_weight_paths = {}
rolling_diag_paths = {}
rolling_bt_paths = {}

for method, weights in rolling_weights.items():
    path = output_dir / f"rolling_weights_{method}.csv"
    weights.to_csv(path)
    rolling_weight_paths[method] = str(path)

for method, diag in rolling_diagnostics.items():
    path = output_dir / f"rolling_diagnostics_{method}.csv"
    diag.to_csv(path, index=False)
    rolling_diag_paths[method] = str(path)

for method, bt in rolling_backtests.items():
    path = output_dir / f"rolling_backtest_{method}.csv"
    bt.to_csv(path)
    rolling_bt_paths[method] = str(path)

manifest = {
    "dataset_name": "hierarchical_risk_parity_outputs",
    "schema_version": "hierarchical_risk_parity_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "asset_cols": asset_cols,
    "split_metadata": split_metadata,
    "scipy_cluster_available": SCIPY_CLUSTER_AVAILABLE,
    "scipy_optimize_available": SCIPY_OPTIMIZE_AVAILABLE,
    "sklearn_available": SKLEARN_AVAILABLE,
    "ordered_assets": ordered_assets,
    "covariance_method": cov_method,
    "covariance_shrinkage": cov_shrinkage,
    "erc_info": erc_info,
    "rolling_performance": rolling_performance.to_dict(orient="records"),
    "noise_summary": noise_summary.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "rolling_weight_files": rolling_weight_paths,
    "rolling_diagnostic_files": rolling_diag_paths,
    "rolling_backtest_files": rolling_bt_paths,
    "notes": [
        "HRP converts correlations into distances using sqrt((1-rho)/2).",
        "Hierarchical clustering is performed with scipy average linkage when available; otherwise a nearest-neighbour fallback order is used.",
        "Recursive bisection allocates capital between clusters using inverse cluster variance.",
        "HRP is compared with equal weight, inverse volatility, GMV, and ERC.",
        "Rolling HRP includes turnover costs and weights are shifted by one day before applying returns.",
        "Noise sensitivity tests perturb the training data and recompute weights.",
        "HRP avoids direct covariance inversion but still depends on correlation and covariance estimation quality."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

returns_path, static_weights_path, rolling_performance_path, governance_flags_path, manifest_path

## 29. Limitations

### 29.1 Synthetic data

The notebook uses synthetic returns with designed cluster structure.

Real market clusters can be unstable and less clean.

### 29.2 Clustering choices matter

Different linkage methods can produce different hierarchies.

### 29.3 HRP is not optimal in a universal sense

HRP is a robust heuristic, not a theorem that guarantees superior returns.

### 29.4 Correlation instability

During crises, correlations can rise and clusters can collapse.

### 29.5 No expected returns

HRP ignores expected returns.

A low-risk cluster can still have poor expected performance.

### 29.6 Simplified transaction costs

Costs are fixed bps.

Real costs depend on liquidity, spread, volatility, market impact, and turnover.

### 29.7 Constraints are basic

The notebook uses long-only weights and does not include leverage, margin, capacity, or regulatory constraints.

### 29.8 Dendrograms can be overinterpreted

A cluster tree is a statistical summary, not a causal map of markets.

## 30. Research frontier and extensions

Important extensions include:

1. **Nested clustered optimisation**  
   Combine HRP clusters with intra-cluster optimisation.

2. **Hierarchical Equal Risk Contribution**  
   Set explicit risk budgets by cluster.

3. **Dynamic clustering**  
   Monitor cluster tree changes through time.

4. **Robust correlation estimates**  
   Use rank correlation, Kendall tau, or robust covariance.

5. **Distance alternatives**  
   Tail dependence, partial correlation, mutual information.

6. **Regime-aware HRP**  
   Different cluster structures in calm and stress regimes.

7. **HRP with expected returns**  
   Blend hierarchy with alpha signals.

8. **Transaction-cost-aware HRP**  
   Penalise rebalancing and cluster instability.

9. **Tail-risk HRP**  
   Cluster using downside correlations or CVaR contributions.

10. **Chinese futures HRP**  
   Cluster metals, energy, agriculture, chemicals, and financial futures using rolling correlation and night-session effects.

## 31. Suggested follow-up notebooks

This notebook naturally leads to:

1. **04_11_black_litterman_portfolio_construction**  
   Blend priors and views.

2. **04_12_correlation_breakdown_and_regime_risk**  
   Study cluster instability in stress.

3. **04_13_portfolio_constraints_margin_and_leverage**  
   Add real implementation constraints.

4. **05_01_vectorized_backtest_engine**  
   Run HRP as a reusable allocation module.

5. **05_05_walk_forward_model_validation**  
   Formal validation of rolling allocation methods.

6. **07_01_multi_asset_cta_research_pipeline**  
   Use HRP in futures allocation.

## 32. Summary

This notebook implemented Hierarchical Risk Parity.

It showed how to:

1. simulate clustered multi-asset returns;
2. estimate covariance and correlation;
3. convert correlation into distance;
4. perform hierarchical clustering;
5. visualise a dendrogram;
6. quasi-diagonalise the correlation matrix;
7. compute inverse-variance cluster risk;
8. allocate weights by recursive bisection;
9. compare HRP to equal weight, inverse volatility, GMV, and ERC;
10. evaluate static out-of-sample performance;
11. run rolling HRP with transaction costs;
12. analyse realised asset and class risk contributions;
13. test sensitivity to covariance noise;
14. evaluate stress-regime behaviour;
15. compare VaR/CVaR;
16. compute diversification ratios;
17. create governance flags and audits.

The key computational takeaway:

> HRP uses hierarchical clustering and recursive bisection to construct covariance-aware portfolios without directly inverting the covariance matrix.

The key financial takeaway:

> HRP can improve allocation stability and cluster diversification, but it still depends on correlation estimates and can fail when crisis correlations change the hierarchy.

## 33. Further reading

- López de Prado, "Building Diversified Portfolios that Outperform Out of Sample."
- López de Prado, *Advances in Financial Machine Learning.*
- J. P. Morgan and institutional literature on portfolio clustering and risk allocation.
- Meucci, *Risk and Asset Allocation.*
- Literature on hierarchical clustering, covariance estimation, HRP, and robust portfolio construction.